# Credit Union Quarterly Data Ingestion

Ingests NCUA call-report data quarter-by-quarter into a Delta table using
**MERGE (upsert)** on the natural key `(cu_number, cycle_date, join_number)`.

- **Idempotent** — run it as many times as you want; no duplicate rows.
- **Incremental** — narrow `start_year`/`end_year` to only fetch new data;
  existing rows are updated in place, new rows are inserted.
- **Schema-evolving** — new NCUA account columns are added automatically.

In [ ]:
import re
from delta.tables import DeltaTable
from ingest_ncua_call_report import iter_quarter_dataframes

# -- Natural key for MERGE (must match the NCUA KEY_COLUMNS after clean_col) --
MERGE_KEYS = ["cu_number", "cycle_date", "join_number"]

def clean_col(name):
    """Sanitize column name for Delta table compatibility."""
    name = name.lower()
    name = re.sub(r"[ ,;{}()\n\t=/]+", "_", name)
    name = name.replace("-", "_")
    name = re.sub(r"_+", "_", name).strip("_")
    return name

def make_unique(cols):
    """Deduplicate column names by appending a counter."""
    seen = {}
    new_cols = []
    for c in cols:
        if c not in seen:
            seen[c] = 0
            new_cols.append(c)
        else:
            seen[c] += 1
            new_cols.append(f"{c}_{seen[c]}")
    return new_cols

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.ncua")

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
# For a full backfill:  start_year=1994, end_year=2026
# For incremental:      start_year=2025, end_year=2026  (only new quarters)
start_year = 1994
end_year = 2026
table_name = "workspace.ncua.ncua_bronze"
download_workers = 8

# Enable Delta schema auto-merge so new NCUA columns are added automatically
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

table_exists = spark.catalog.tableExists(table_name)
merge_condition = " AND ".join(
    f"target.{k} = source.{k}" for k in MERGE_KEYS
)
quarters_written = 0
quarters_failed = 0

for quarter, pdf in iter_quarter_dataframes(
    start_year=start_year,
    end_year=end_year,
    download_workers=download_workers,
):
    # Cast all columns to string for schema consistency across decades
    pdf = pdf.astype(str)

    # Clean column names for Delta compatibility
    cleaned = [clean_col(c) for c in pdf.columns]
    pdf.columns = make_unique(cleaned)

    try:
        sdf = spark.createDataFrame(pdf)

        if not table_exists:
            # First-ever run: create the table
            (
                sdf.write
                .format("delta")
                .mode("overwrite")
                .option("mergeSchema", "true")
                .saveAsTable(table_name)
            )
            table_exists = True
            print(f"  -> CREATED table with {len(pdf):,} rows from {quarter}")
        else:
            # MERGE: update existing rows, insert new ones
            delta_table = DeltaTable.forName(spark, table_name)
            (
                delta_table.alias("target")
                .merge(sdf.alias("source"), merge_condition)
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )
            print(f"  -> MERGED {len(pdf):,} rows for {quarter}")

        quarters_written += 1
    except Exception as e:
        quarters_failed += 1
        print(f"  -> FAILED {quarter}: {e}")
    finally:
        del pdf, sdf

print(f"\nIngestion complete. {quarters_written} quarters written, {quarters_failed} failed.")

In [ ]:
# ── Verification ───────────────────────────────────────────────────────────
display(spark.sql(f"SELECT COUNT(*) as total_rows FROM {table_name}"))
display(spark.sql(f"SELECT DISTINCT cycle_date FROM {table_name} ORDER BY cycle_date"))
# Check for duplicates on the natural key (should return 0)
display(spark.sql(f"""
    SELECT cu_number, cycle_date, join_number, COUNT(*) as cnt
    FROM {table_name}
    GROUP BY cu_number, cycle_date, join_number
    HAVING cnt > 1
    LIMIT 10
"""))